# Phase 0 — HDC validation

Purpose: empirically verify the three load-bearing properties of the Kanerva substrate **before** writing a single line of production code.

Properties under test:

1. **Fuzzy content-addressing** — a noisy cue still retrieves the correct stored item up to a substantial noise floor.
2. **Holographic degradation** — adding noise across all stored items degrades recall smoothly rather than catastrophically.
3. **Compositionality** — `bind`/`unbind` lets us encode role-filler pairs and recover the filler given the role.

All experiments use fixed seeds so results are reproducible across runs. CI executes this notebook end-to-end on every PR; breaking the math breaks the build.

> Note: this notebook validates **properties**. Phase 1's TypeScript reference implementation will follow `SPEC.md` exactly (BLAKE3-based `randomHV`, deterministic tiebreakers, etc.). The Python primitives here use stdlib `blake2b` for portability and are not byte-equivalent to the spec — they're a faithful model of the math, not the wire format.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hashlib import blake2b

np.random.seed(42)

D = 10000  # hypervector dimension

## 1. Primitives

- `random_hv(seed)` — deterministic random binary hypervector from a byte seed.
- `bundle(hvs)` — element-wise majority over a set of hypervectors. Ties broken by a seeded RNG (the spec uses a BLAKE3-derived tiebreaker; here we use numpy's default RNG with a fixed seed for portability).
- `bind(a, b)` — element-wise XOR. Self-inverse.
- `similarity(a, b)` — `1 - hamming_distance / D`, in `[0, 1]`.

In [ ]:
def random_hv(seed: bytes, d: int = D) -> np.ndarray:
    """Deterministic random binary hypervector from a seed."""
    digest_seed = int.from_bytes(blake2b(seed, digest_size=8).digest(), 'big')
    rng = np.random.default_rng(digest_seed)
    return rng.integers(0, 2, size=d, dtype=np.uint8)


def bundle(hvs: list) -> np.ndarray:
    """Element-wise majority. Ties broken by a seeded coinflip."""
    if not hvs:
        raise ValueError("empty bundle")
    arr = np.stack(hvs)
    sums = arr.sum(axis=0)
    n = arr.shape[0]
    result = (sums > n / 2).astype(np.uint8)
    if n % 2 == 0:
        ties = (sums == n // 2)
        tie_rng = np.random.default_rng(0xC0DE)
        result[ties] = tie_rng.integers(0, 2, size=int(ties.sum()))
    return result


def bind(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.bitwise_xor(a, b)


def similarity(a: np.ndarray, b: np.ndarray) -> float:
    return 1.0 - (np.count_nonzero(a != b) / len(a))

In [ ]:
# Sanity checks
a = random_hv(b"hello")
b = random_hv(b"world")

print(f"sim(a, a)                = {similarity(a, a):.4f}   (expect 1.0)")
print(f"sim(a, b)                = {similarity(a, b):.4f}   (expect ~0.5)")
print(f"sim(bind(a,b), b)        = {similarity(bind(a, b), b):.4f}   (expect ~0.5)")
print(f"sim(bind(bind(a,b),b),a) = {similarity(bind(bind(a, b), b), a):.4f}   (expect 1.0 — XOR is self-inverse)")

assert similarity(a, a) == 1.0
assert 0.45 < similarity(a, b) < 0.55
assert similarity(bind(bind(a, b), b), a) == 1.0

## 2. Property 1 — Fuzzy content-addressing

Store 1000 random items. For each noise level `p` from `0` to `0.5`, generate a noisy cue from a randomly-chosen target by flipping each bit with probability `p`, then check whether the nearest stored item (by similarity) is the original target.

Expected behavior: near-perfect recall up to ~0.3 bit-flip rate, sharp falloff approaching the theoretical 0.5 limit (where the cue becomes random).

In [ ]:
def add_noise(hv: np.ndarray, p: float, rng: np.random.Generator) -> np.ndarray:
    flips = rng.random(len(hv)) < p
    return np.bitwise_xor(hv, flips.astype(np.uint8))


N_ITEMS = 1000
items = np.stack([random_hv(f"item_{i}".encode()) for i in range(N_ITEMS)])  # (N, D)

noise_levels = np.linspace(0, 0.5, 26)
trials_per_level = 200
exp_rng = np.random.default_rng(123)

accuracies = []
for p in noise_levels:
    correct = 0
    for _ in range(trials_per_level):
        target = int(exp_rng.integers(N_ITEMS))
        cue = add_noise(items[target], p, exp_rng)
        # vectorized similarity
        ham = np.count_nonzero(items != cue, axis=1)
        sims = 1.0 - ham / D
        best = int(np.argmax(sims))
        if best == target:
            correct += 1
    accuracies.append(correct / trials_per_level)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(noise_levels, accuracies, marker='o', color='C0')
ax.set(xlabel="Bit-flip probability (cue noise)",
       ylabel="Recall@1 accuracy",
       title=f"Fuzzy recall under cue noise (D={D}, N={N_ITEMS})")
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
ax.axhline(y=1/N_ITEMS, linestyle='--', color='gray', label=f'chance (1/{N_ITEMS})')
ax.legend()
plt.tight_layout()
plt.show()

# Conformance: recall@1 should be near-perfect at low noise.
assert accuracies[0] == 1.0, "noise=0 should be perfect recall"
assert accuracies[10] > 0.95, f"noise=0.2 expected >95% recall, got {accuracies[10]:.3f}"

## 3. Property 2 — Holographic degradation

Add noise to **every stored item** at increasing dropout rates. Query with a clean cue. Show that recall degrades smoothly — every item suffers a little, no specific item dies.

This is the property that distinguishes a holographic substrate from a block-addressed one. In an SSD, knocking out a sector destroys whatever data lived there. In a Kanerva substrate, knocking out `d%` of bits across the whole substrate causes every recall similarity to drop by `~d`, but the *ranking* of similarities is mostly preserved — so recall@1 holds up far longer than a worst-case loss model would predict.

In [ ]:
dropout_rates = np.linspace(0, 0.45, 19)
trials = 200
deg_rng = np.random.default_rng(456)

recall_at_1 = []
mean_target_sim = []
for d in dropout_rates:
    # Flip d-fraction of bits in every stored item — substrate-wide noise
    flip_mask = (deg_rng.random(items.shape) < d).astype(np.uint8)
    degraded = np.bitwise_xor(items, flip_mask)

    correct = 0
    sims_to_target = []
    for _ in range(trials):
        target = int(deg_rng.integers(N_ITEMS))
        cue = items[target]  # clean cue
        ham = np.count_nonzero(degraded != cue, axis=1)
        sims = 1.0 - ham / D
        best = int(np.argmax(sims))
        if best == target:
            correct += 1
        sims_to_target.append(sims[target])
    recall_at_1.append(correct / trials)
    mean_target_sim.append(float(np.mean(sims_to_target)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(dropout_rates, mean_target_sim, marker='o', color='C0')
axes[0].set(xlabel="Substrate noise fraction",
            ylabel="Mean sim(clean cue, noisy target)",
            title="Cue–target similarity decays linearly")
axes[0].grid(alpha=0.3)
axes[1].plot(dropout_rates, recall_at_1, marker='o', color='C1')
axes[1].set(xlabel="Substrate noise fraction",
            ylabel="Recall@1",
            title="Recall accuracy under substrate corruption")
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Conformance: recall holds up well into substantial substrate noise.
assert recall_at_1[0] == 1.0
assert recall_at_1[8] > 0.9, f"~20% substrate noise: expected >90% recall, got {recall_at_1[8]:.3f}"

## 4. Property 3 — Compositionality

Bind role–filler pairs (`name`↔`alice`, `age`↔`30`, `city`↔`boston`), bundle them into a single hypervector that represents the whole record, then unbind a single role from the record to recover the filler.

Recovery is *approximate* — the bundled record contains the other role-filler pairs as noise — but the cleanup memory (the set of known fillers) reliably picks the right one.

This is what makes Kanerva structured: you can store relational data in a single vector and query it by relational predicate, all in HD space.

In [ ]:
roles = {r: random_hv(f"role:{r}".encode()) for r in ["name", "age", "city"]}
fillers = {f: random_hv(f"filler:{f}".encode())
           for f in ["alice", "bob", "carol", "30", "25", "40", "boston", "chicago", "denver"]}

record = bundle([
    bind(roles["name"], fillers["alice"]),
    bind(roles["age"], fillers["30"]),
    bind(roles["city"], fillers["boston"]),
])

print("Recovering values from a single bundled record:\n")
for role_name, role_hv in roles.items():
    query = bind(record, role_hv)
    sims = {fname: similarity(fhv, query) for fname, fhv in fillers.items()}
    best = max(sims, key=sims.get)
    sorted_sims = sorted(sims.items(), key=lambda kv: -kv[1])
    print(f"  unbind(record, {role_name!r:10}) -> {best!r:10} (sim={sims[best]:.3f})")
    runners_up = ', '.join(f'{k}={v:.3f}' for k, v in sorted_sims[1:4])
    print(f"      runners-up: {runners_up}\n")

expected = {"name": "alice", "age": "30", "city": "boston"}
for role_name, role_hv in roles.items():
    query = bind(record, role_hv)
    sims = {fname: similarity(fhv, query) for fname, fhv in fillers.items()}
    best = max(sims, key=sims.get)
    assert best == expected[role_name], (
        f"compositional recovery failed for {role_name}: got {best!r}"
    )

## 5. Conclusions

All three properties hold empirically at `D = 10000`:

1. **Fuzzy recall**: perfect through `~0.2` bit-flip noise; graceful falloff to the theoretical limit at `~0.5`.
2. **Holographic degradation**: recall@1 stays above 90% even at 20% substrate noise. The math degrades like a hologram, not like a disk.
3. **Compositionality**: bound role-filler pairs survive bundling and unbind correctly through the cleanup memory.

These are the empirical foundations on which the Phase 1 TypeScript reference implementation will be built. The spec (`SPEC.md`) is now locked at draft v0.1.0 pending one final read.

**Open questions to resolve before SPEC.md is frozen** (tracked in `SPEC.md` §11):

- Default `D`: 8192 (SIMD-aligned) vs 10000 (literature default).
- Cleanup-memory implementation: brute-force first, LSH for v0.2.0.
- Embedding-to-HV encoding scheme: thermometer + projection (current proposal) vs random-Fourier-features.

Next: write the TypeScript reference implementation in `packages/core-ts/` against this spec.